# RAG Smoke Test

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

In [ ]:
import json

import sys
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Ensure repo root is on path when running from notebooks/
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Set DOCS_DIR relative to repo root if not already set
if not os.environ.get("DOCS_DIR"):
    os.environ["DOCS_DIR"] = str(repo_root / "docs")

In [ ]:
articles = [
    "Artículo 1. Transferencias internacionales. Las personas jurídicas que realicen transferencias internacionales deberán informar a la Unidad de Información Financiera dentro de los plazos establecidos.",
    "Artículo 2. Personas políticamente expuestas. Se consideran personas políticamente expuestas (PEP) a quienes desempeñen o hayan desempeñado funciones públicas prominentes en el país o en el extranjero.",
    "Artículo 3. Obligaciones de reporte. Los sujetos obligados deberán reportar toda operación sospechosa de lavado de activos o financiamiento del terrorismo ante la autoridad competente.",
]

In [ ]:
from tools.postgresql_tools import clear_table
from tools.elasticsearch_tools import clear_index
from tools.falkordb_tools import clear_graph

def clear_all_stores():
    clear_table()
    clear_index()
    clear_graph()
    print("All stores cleared")

In [ ]:
import psycopg2, os

def reset_db_just_in_case():
    conn = psycopg2.connect(
        host=os.environ["POSTGRES_HOST"],
        port=int(os.environ.get("POSTGRES_PORT", 5432)),
        dbname="postgres",          # conectarse a la DB de sistema, no a la tuya
        user=os.environ["POSTGRES_USER"],
        password=os.environ["POSTGRES_PASSWORD"],
    )
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("""
            SELECT pg_terminate_backend(pid)
            FROM pg_stat_activity
            WHERE datname = %s AND pid <> pg_backend_pid()
        """, (os.environ["POSTGRES_DB"],))
        print(cur.fetchall())
    conn.close()

# Individual modules test

In [ ]:
clear_all_stores()

## Dense Module

In [ ]:
from rag import dense

In [ ]:
dense.initialize()
dense.index("test_doc", articles, country_code="CO")

In [ ]:
dense_results = dense.search("transferencia internacional persona jurídica", top_k=3, country_code="CO")
print(json.dumps(dense_results, indent=2, ensure_ascii=False))

## Sparse Module

In [ ]:
from rag import sparse

In [ ]:
sparse.initialize()
sparse.index("test_doc", articles, country_code="CO")

In [ ]:
sparse_results = sparse.search("transferencia internacional persona jurídica", top_k=3, country_code="CO")
print(json.dumps(sparse_results, indent=2, ensure_ascii=False))

## Graph Module

In [ ]:
from rag.graph import GraphLayer

In [ ]:
articles_doc1 = [
    "Artículo 1. Transferencias internacionales. Las personas jurídicas que realicen transferencias internacionales deberán informar a la Unidad de Información Financiera dentro de los plazos establecidos.",
    "Artículo 2. Personas políticamente expuestas. Se consideran personas políticamente expuestas (PEP) a quienes desempeñen o hayan desempeñado funciones públicas prominentes en el país o en el extranjero.",
    "Artículo 3. Obligaciones de reporte. Los sujetos obligados deberán reportar toda operación sospechosa de lavado de activos o financiamiento del terrorismo ante la autoridad competente.",
]

articles_doc2 = [
    "Artículo 1. Umbrales de reporte. Conforme a lo establecido en la circular 001/2024 UIAF Art. 1, los umbrales para el reporte de transferencias internacionales serán actualizados anualmente por la autoridad competente.",
    "Artículo 2. Debida diligencia. Los sujetos obligados deberán aplicar medidas de debida diligencia reforzada respecto de clientes que presenten señales de alerta asociadas al lavado de activos.",
    "Artículo 3. Registros de operaciones. Los reportes de operaciones en efectivo deberán conservarse por un período mínimo de cinco años y estar disponibles para consulta de la autoridad supervisora.",
]

In [ ]:
graph = GraphLayer()

doc1 = {
    "document_id": "test_doc",
    "country": "Colombia",
    "country_code": "CO",
    "issuer": "UIAF",
    "type": "circular",
    "year": "2024",
    "number": "001",
    "title": "umbrales reporte financiero",
}
doc2 = {
    "document_id": "test_doc_2",
    "country": "Colombia",
    "country_code": "CO",
    "issuer": "UIAF",
    "type": "resolucion",
    "year": "2024",
    "number": "042",
    "title": "debida diligencia y registros de operaciones",
}
graph.initialize()
graph.load_catalog([doc1, doc2])

graph.index("test_doc", "\n\n".join(articles_doc1), "CO")
graph.index("test_doc_2", "\n\n".join(articles_doc2), "CO")

In [ ]:
graph_results = graph.expand("test_doc_2")
print("Expand test_doc_2:", graph_results)

# Full Indexing Pipeline

In [ ]:
clear_all_stores()

In [ ]:
from rag.indexer import index_documents

In [ ]:
index_documents(
    before = lambda doc: print('Starting with %s' % doc['title']),
    after  = lambda doc: print('Finished with %s' % doc['title']),
)
print("Indexing complete")

## Dense Retrieval

In [ ]:
from rag import dense

In [ ]:
dense_results = dense.search("transferencia internacional persona jurídica", top_k=3, country_code="CO")
print(json.dumps(dense_results, indent=2, ensure_ascii=False))

## Sparse Retrieval

In [ ]:
from rag import sparse

In [ ]:
sparse_results = sparse.search("transferencia internacional persona jurídica", top_k=3, country_code="CO")
print(json.dumps(sparse_results, indent=2, ensure_ascii=False))

## Comparison

In [ ]:
query = "persona políticamente expuesta PEP obligaciones reporte"

dense_res = dense.search(query, top_k=5, country_code="CO")
sparse_res = sparse.search(query, top_k=5, country_code="CO")

overlap = set(dense_res) & set(sparse_res)

print(f"Dense results : {len(dense_res)}")
print(f"Sparse results: {len(sparse_res)}")
print(f"Overlap       : {len(overlap)} — {overlap if overlap else 'none'}")

## Hybrid Retrieval

In [ ]:
from rag import retriever
retriever.initialize()

In [ ]:
query = "persona políticamente expuesta PEP obligaciones reporte"

hybrid_results = retriever.hybrid_retrieve(query, top_k=1, country_code="CO")

print("Hybrid results:")
for i, doc_id in enumerate(hybrid_results, 1):
    print(f"  {i}. {doc_id}")

print(f"\nDense only : {dense_res}")
print(f"Sparse only: {sparse_res}")
print(f"Hybrid     : {hybrid_results}")